# Step 4B: Mean Reversion Strategy Signal Construction

This notebook implements and demonstrates Step 4B of the quantitative research pipeline: **Mean Reversion Strategy Signal Construction**.

### Objective
Construct a reusable rolling Z-score mean reversion signal generator. Today's close price is used to generate today's signal, which is executed tomorrow to ensure look-ahead bias prevention. We use a finite-state machine (FSM) to handle positions and exit thresholds, extract trades, and analyze signal statistics.

## Mean Reversion Theory

### What is Mean Reversion?
Mean reversion is a financial theory suggesting that asset prices and historical returns tend to revert back to their long-term average or mean. When the price of an asset deviates significantly from its historical average, it is expected to move back toward that average over time.

### Why Financial Prices Revert
- **Liquidity Demands & Market Microstructure**: Large institutional orders can temporarily push asset prices away from their fair values due to imbalance in order books. Once the liquidity pressure subsides, prices revert.
- **Overreaction and Behavior**: Investors often overreact to news events (earnings, macroeconomic announcements), causing prices to overshoot. Corrective forces eventually bring prices back to the mean.
- **Fundamental Value Anchoring**: For indices or large companies, valuation anchors (earnings, assets) restrict the price from drifting indefinitely. Economic cycles dictate that extreme profits or losses revert to industry averages.

### Momentum vs. Mean Reversion
- **Momentum (Trend Following)**: Assumes that trends persist. It buys high and sells higher (or shorts low and covers lower). It works well in trending, high-regime markets.
- **Mean Reversion**: Assumes that price deviations are temporary and self-correcting. It buys low (undervalued relative to historical average) and sells when the price reverts. It works well in range-bound, choppy, or low-trend markets.

### Markets and Failure Cases
- **Performs Well**: Range-bound markets, equity indices (e.g., Nifty 50, S&P 500) where component rebalancing maintains index stability, commodities, and currency pairs (which naturally trade in ranges due to monetary policy constraints).
- **Fails (The "Value Trap")**: Strong trending markets, secular growth stocks (which continue upwards), bankrupt stocks (which drop to zero without reverting), and major corporate restructuring events where the asset's fundamental mean changes permanently.

### Why Statistical Normalization (Z-Score) is Required
Raw price differences (e.g. Price - SMA) are not comparable across different time periods because of price drift. If an index goes from 5,000 to 25,000, a 200-point deviation is massive at 5,000 but negligible at 25,000. 

By dividing the deviation by the rolling standard deviation, we normalize the deviation into unit standard deviations (Z-score). This makes thresholds like $-2.0$ statistically comparable across time and different assets regardless of their nominal values.

In [ ]:
import os
import sys

# Insert project source root to system path for importing local modules
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.mean_reversion import MeanReversionSignalGenerator
from src.validators import validate_mean_reversion_params, validate_signals

# Setup matplotlib rendering style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Setup complete. Modules imported successfully.")

## Section 1: Load and Validate Dataset

We load `data/processed/nifty50_clean.parquet` and perform validations to verify the index, Close column, and missing values.

In [ ]:
parquet_path = "../data/processed/nifty50_clean.parquet"

# Verify file existence
if not os.path.exists(parquet_path):
    raise FileNotFoundError(f"Clean parquet file not found at: {parquet_path}")

df = pd.read_parquet(parquet_path)
print(f"Loaded clean dataset. Rows: {len(df)}, Columns: {list(df.columns)}")

# Validate constraints
assert isinstance(df.index, pd.DatetimeIndex), "Index must be a DatetimeIndex."
assert "Close" in df.columns, "'Close' column must exist in the dataset."
assert df["Close"].isna().sum() == 0, "'Close' column contains missing values."
print("Dataset structural validations passed successfully.")

## Section 2: Indicator Construction & Signal Generation

We instantiate the `MeanReversionSignalGenerator` with:
- Rolling Window = 20
- Entry Threshold = -2.0
- Exit Threshold = -0.5

The signal generator will compute rolling mean, standard deviation, Z-score, raw signals, execution signals, and positions.

In [ ]:
window = 20
entry_threshold = -2.0
exit_threshold = -0.5

generator = MeanReversionSignalGenerator(
    window=window, 
    entry_threshold=entry_threshold, 
    exit_threshold=exit_threshold
)

# Generate signals and detect trades
signals_df = generator.generate_signals(df, close_col="Close")
trades_df = generator.detect_trades(signals_df, position_col="Position")
stats = generator.compute_statistics(signals_df, trades_df)

print("\n=== Default Strategy Statistics ===")
for key, val in stats.items():
    if isinstance(val, float):
        print(f"{key:45}: {val:.4f}")
    else:
        print(f"{key:45}: {val}")

## Section 3: Look-Ahead Bias Prevention and Validation

### Why today's Close cannot be traded today
To determine the rolling Z-score at day $t$, we need the closing price of day $t$. The closing price is only determined at the exact moment the market closes. Because of this, it is physically impossible to enter a position at the day $t$ closing price based on that day's signal. 

If we simulate trading on day $t$ using day $t$'s close signal, we introduce **Look-Ahead Bias** (or future leakage), making our backtest artificially profitable.

To resolve this, we shift the raw signal forward by 1 trading day:
$$\text{Execution\_Signal}_t = \text{Raw\_Signal}_{t-1}$$
This ensures point-in-time correctness: the signal calculated using day $t-1$'s closing data is executed on day $t$ (modeled as holding the position from the close of day $t$ onward, or entering at the close of day $t$).

In [ ]:
# Let's visualize the 1-day execution lag by zooming in on a small window
sample_slice = signals_df.loc['2023-05-01':'2023-08-31']

fig, ax = plt.subplots(figsize=(14, 5))
ax.step(sample_slice.index, sample_slice['Raw_Signal'], where='pre', label='Raw Signal (t)', color='blue', linestyle='--', alpha=0.6)
ax.step(sample_slice.index, sample_slice['Execution_Signal'], where='pre', label='Execution Signal (t+1)', color='red', linewidth=2)
ax.set_title("Look-Ahead Bias Prevention: 1-Day Execution Lag (Zoomed-In 2023 Window)", fontweight='bold')
ax.set_xlabel("Date")
ax.set_ylabel("Signal State")
ax.set_ylim(-0.1, 1.1)
ax.legend(loc='upper right')
ax.grid(True, linestyle=':', alpha=0.6)
plt.show()

## Section 4: Trade Detection

Trades are defined by position transitions:
- BUY trade (Entry) occurs when Position goes from `0 -> 1`.
- SELL trade (Exit) occurs when Position goes from `1 -> 0`.

We print the list of detected trades and verify that holding periods and exit rules are logically sound.

In [ ]:
print(f"Total trades detected: {len(trades_df)}")
if not trades_df.empty:
    display(trades_df.head(10))
else:
    print("No trades detected in this dataset.")

## Section 5: Professional Visualizations

We generate the 5 publication-quality figures requested in Section 11 and verify they are saved in `reports/figures/`.

In [ ]:
figures_dir = "../reports/figures"
os.makedirs(figures_dir, exist_ok=True)

# Generate all performance plots programmatically
generator.plot_performance(signals_df, trades_df, output_dir=figures_dir)
print(f"All charts saved successfully to: {os.path.abspath(figures_dir)}")

## Section 6: Sensitivity Analysis

We perform a grid-search analysis on different parameter combinations:
- Rolling Window: `10`, `20`, `50`
- Entry Threshold: `-1.5`, `-2.0`, `-2.5`
- Exit Threshold: Fixed at `-0.5`

We compare: the number of trades, average holding period (trading days), and percentage of time invested. We do NOT evaluate profitability.

In [ ]:
windows = [10, 20, 50]
entry_thresholds = [-1.5, -2.0, -2.5]
fixed_exit = -0.5

sensitivity_results = []

for w in windows:
    for et in entry_thresholds:
        g = MeanReversionSignalGenerator(window=w, entry_threshold=et, exit_threshold=fixed_exit)
        sig_df = g.generate_signals(df, close_col="Close")
        tr_df = g.detect_trades(sig_df)
        st = g.compute_statistics(sig_df, tr_df)
        
        sensitivity_results.append({
            "Rolling Window": w,
            "Entry Threshold": et,
            "Exit Threshold": fixed_exit,
            "Number of Trades": st.get("Number_of_Trades", 0),
            "Average Hold (Trading Days)": round(st.get("Average_Holding_Period_Trading_Days", 0), 2),
            "Percentage Time Invested (%)": round(st.get("Percentage_of_Time_Invested", 0), 2)
        })

sensitivity_df = pd.DataFrame(sensitivity_results)
print("=== Parameter Sensitivity Analysis Matrix ===")
display(sensitivity_df)

### Interpretation of Parameter Sensitivity Results
1. **Effect of Rolling Window size**:
   - A shorter rolling window (e.g. 10 days) responds quickly to price movements. It computes the mean and standard deviation over a shorter history, meaning Z-scores fluctuate rapidly. This typically results in a higher number of trades but shorter holding periods.
   - A longer rolling window (e.g. 50 days) represents a slower-moving historical average. Deviations take longer to form and resolve. This results in fewer trades and longer holding periods.
2. **Effect of Entry Threshold**:
   - A looser entry threshold (e.g. -1.5) is crossed more frequently. The strategy enters trades easily, resulting in a higher number of trades and higher percentage of time invested.
   - A tighter, more conservative entry threshold (e.g. -2.5) represents extreme deviations (lower probability events). The strategy enters trades less frequently, resulting in fewer trades and lower time invested.

## Section 7: Export Signals

We export the processed signal dataset to `data/processed/mean_reversion_signals.parquet` and `data/processed/mean_reversion_signals.csv`.

In [ ]:
output_parquet_path = "../data/processed/mean_reversion_signals.parquet"
output_csv_path = "../data/processed/mean_reversion_signals.csv"

signals_df.to_parquet(output_parquet_path, index=True, engine='pyarrow')
signals_df.to_csv(output_csv_path, index=True)

print(f"Saved mean reversion signals to Parquet: {output_parquet_path}")
print(f"Saved mean reversion signals to CSV: {output_csv_path}")